In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/evaluator/evaluator.py
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet
/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/g_i_warm.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/g_u_warm.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-warm/user_index_warm.json
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/ngocccchannn/cold-sc

In [2]:
import copy
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import sys

# Set seed cố định
def set_seed(s=42):
  random.seed(s)
  np.random.seed(s)
  torch.manual_seed(s)
  torch.cuda.manual_seed_all(s)
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

sys.path.append('/kaggle/input/datasets/ngocccchannn/evaluator') 
from evaluator import evaluate

Sử dụng thiết bị: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [3]:
# Nạp các dữ liệu cần thiết 
raw_course_text_embeddings = np.load('/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_course_text_embeddings.npy')
print(f'Kích thước của embedding văn bản gốc: {raw_course_text_embeddings.shape}')

with open('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/item_index.json', 'r') as f:
    course_ids = json.load(f)
assert len(course_ids) == 3769 and course_ids == sorted(course_ids), "Lỗi item_index.json"
assert course_ids == json.load(open("/kaggle/input/datasets/ngocccchannn/raw-embeddings/raw_embeddings_course_index.json"))["course_id"]

with open('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/user_index_cold.json', 'r') as f:
    user_cold_ids = json.load(f)
    
print(f'Tổng số khóa học: {len(course_ids)}, Tổng số học viên (khóa lạnh): {len(user_cold_ids)}')

g_i_cold = np.load('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_i.npy')
g_u_cold = np.load('/kaggle/input/datasets/ngocccchannn/lightgcn-backbone-cold/g_u.npy')
print(f"Bảng kích thước: g_u_cold = {g_u_cold.shape}, g_i_cold = {g_i_cold.shape}")

assert g_u_cold.shape == (len(user_cold_ids), 64) and g_i_cold.shape == (len(course_ids), 64), "Lỗi kích thước embedding nền chung"

Kích thước của embedding văn bản gốc: (3769, 384)
Tổng số khóa học: 3769, Tổng số học viên (khóa lạnh): 98220
Bảng kích thước: g_u_cold = (98220, 64), g_i_cold = (3769, 64)


In [4]:
# Chuẩn bị dữ liệu đánh giá tập test_cold
te_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/test_cold.parquet')
tr_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/train_cold.parquet')
va_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/cold-scenario-data/val_cold.parquet')

user2id = {str(u): i for i, u in enumerate(user_cold_ids)}
item2id = {str(c): i for i, c in enumerate(course_ids)}

test_target_dict = {}
for u, group in te_cold.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            test_target_dict[u_idx] = item_idxs
print(f'test_target_dict với {len(test_target_dict)} học viên.')

train_hist_dict = {}
for u, group in tr_cold.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            train_hist_dict[u_idx] = item_idxs
print(f'train_hist_dict với {len(train_hist_dict)} học viên.')

val_target_dict = {}
for u, group in va_cold.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            val_target_dict[u_idx] = item_idxs
print(f"val_target_dict với {len(val_target_dict)} học viên.")

test_target_dict với 3340 học viên.
train_hist_dict với 98220 học viên.
val_target_dict với 2092 học viên.


In [5]:
with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_val.json', 'r') as f:
    cold_val_ids = json.load(f)
with open('/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json', 'r') as f:
    cold_test_ids = json.load(f)

cold_val_ids = cold_val_ids.get("cold_items", cold_val_ids) if isinstance(cold_val_ids, dict) else cold_val_ids
cold_test_ids = cold_test_ids.get("cold_items", cold_test_ids) if isinstance(cold_test_ids, dict) else cold_test_ids

cold_item_set = set(str(c) for c in cold_val_ids + cold_test_ids)
cold_idx = np.array(sorted(item2id[c] for c in cold_item_set if c in item2id))

# Kiểm tra assertion khóa lạnh bằng 0 - không có tín hiệu ID 
assert np.allclose(g_i_cold[cold_idx], 0.0), "Lỗi: Khóa lạnh phải có embedding g_i = 0 đối với mô hình thuần ID!"

In [6]:
# Khởi tạo dữ liệu tensor cho DropoutNet
import torch.nn.functional as F

# Chuyển đổi các ma trận numpy sang torch.tensor và đưa lên thiết bị (GPU/CPU)
U_tensor = torch.tensor(g_u_cold, dtype=torch.float32, device=DEVICE)  # (num_users, 64)
V_tensor = torch.tensor(g_i_cold, dtype=torch.float32, device=DEVICE)  # (3769, 64)
Phi_V_tensor = torch.tensor(raw_course_text_embeddings, dtype=torch.float32, device=DEVICE) # (3769, d_content)

print(f"Kích thước tensor U: {U_tensor.shape}")
print(f"Kích thước tensor V: {V_tensor.shape}")
print(f"Kích thước tensor Phi_V (văn bản): {Phi_V_tensor.shape}")

Kích thước tensor U: torch.Size([98220, 64])
Kích thước tensor V: torch.Size([3769, 64])
Kích thước tensor Phi_V (văn bản): torch.Size([3769, 384])


In [7]:
# Định nghĩa lớp DropoutNetItemCold
class DropoutNetItemCold(nn.Module):
    def __init__(self, d=64, d_content=384, h=200):
        super().__init__()
        self.fU = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, d))
        self.fV = nn.Sequential(nn.Linear(d + d_content, h), nn.Tanh(), nn.Linear(h, d))
        
    def forward(self, u_pref, v_pref, v_content):
        # u_pref: embedding học viên từ LightGCN nền
        # v_pref: embedding khóa học từ LightGCN nền (sẽ bị dropout về 0 khi mô phỏng khóa lạnh)
        # v_content: embedding văn bản gốc của khóa học
        return self.fU(u_pref), self.fV(torch.cat([v_pref, v_content], -1))

print("Đã định nghĩa xong class DropoutNetItemCold.")

Đã định nghĩa xong class DropoutNetItemCold.


In [8]:
import itertools, random

space = dict(lr=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2], 
             p=[0.1, 0.3, 0.5, 0.8],
             lam=[0.01, 0.05, 0.1, 0.5], 
             wd=[1e-5, 1e-4, 1e-3])

# Tạo lưới không gian tìm kiếm chuẩn
grid = [dict(zip(space, v)) for v in itertools.product(*space.values())]

# Lấy ngẫu nhiên đúng 20 cấu hình với seed 42 cho mọi mô hình theo Phụ lục A.2
trials = random.Random(42).sample(grid, 20) 
print(f"Đã khởi tạo thành công {len(trials)} cấu hình tinh chỉnh ngẫu nhiên.")

Đã khởi tạo thành công 20 cấu hình tinh chỉnh ngẫu nhiên.


In [9]:
cand_cold_indices = [item2id[str(c)] for c in cold_test_ids if str(c) in item2id]
cand_bool_mask_a = np.zeros(3769, dtype=bool)
if len(cand_cold_indices) > 0:
    cand_bool_mask_a[cand_cold_indices] = True

cand_cold_indices_val = [item2id[str(c)] for c in cold_val_ids if str(c) in item2id]
cand_bool_mask_a_val = np.zeros(3769, dtype=bool)
if len(cand_cold_indices_val) > 0:
    cand_bool_mask_a_val[cand_cold_indices_val] = True

cand_bool_mask_b = np.ones(3769, dtype=bool)

In [10]:
best_val_ndcg = -1.0
best_cfg = None
best_model_state = None

print("--- BẮT ĐẦU QUÁ TRÌNH TINH CHỈNH 20 CẤU HÌNH CHO DROPOUTNET ---")

for idx, cfg in enumerate(trials):
    set_seed(42) # Cố định seed cho mỗi lần khởi tạo cấu hình
    
    model = DropoutNetItemCold(d=64, d_content=Phi_V_tensor.shape[1], h=200).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    
    EPOCHS_TUNING = 30 # Ngân sách epoch cho mỗi trial tinh chỉnh
    
    # Huấn luyện mô hình với tỷ lệ p mô phỏng khóa lạnh trong cfg
    model.train()
    for epoch in range(EPOCHS_TUNING):
        optimizer.zero_grad()
        
        # Mô phỏng khóa lạnh với xác suất p từ cấu hình
        mask = (torch.rand(V_tensor.shape[0], 1, device=DEVICE) > cfg['p']).float()
        v_pref_masked = V_tensor * mask
        
        u_pred, v_pred = model(U_tensor, v_pref_masked, Phi_V_tensor)
        
        orig_scores = torch.matmul(U_tensor, V_tensor.t())
        pred_scores = torch.matmul(u_pred, v_pred.t())
        
        loss = F.mse_loss(pred_scores, orig_scores)
        loss.backward()
        optimizer.step()
        
    # Đánh giá nhanh trên tập cold-val (sử dụng hàm evaluate chung đã import)
    model.eval()
    with torch.no_grad():
        # Khi suy luận khóa lạnh: v_pref = 0, hệ thống chỉ dùng đặc trưng nội dung v_content (Phụ lục A.3)
        v_pref_cold_sim = torch.zeros_like(V_tensor)
        u_eval, v_eval = model(U_tensor, v_pref_cold_sim, Phi_V_tensor)
        
        def score_fn(users_tensor):
            u_idx = users_tensor.to(DEVICE)
            return torch.matmul(u_eval[u_idx], v_eval.t())
            
        val_res = evaluate(score_fn, val_target_dict, mask=train_hist_dict, cand=cand_bool_mask_a_val, ks=(20,), bs=1024)
        current_ndcg = val_res['NDCG@20'].mean()
        
    print(f"Trial [{idx+1}/20] | Config: {cfg} | NDCG@20 (Val): {current_ndcg:.4f}")
    
    if current_ndcg > best_val_ndcg:
        best_val_ndcg = current_ndcg
        best_cfg = cfg
        best_model_state = copy.deepcopy(model.state_dict())

print(f"\n--- TINH CHỈNH HOÀN TẤT ---")
print(f"Cấu hình tốt nhất: {best_cfg}")
print(f"NDCG@20 cao nhất trên tập Val: {best_val_ndcg:.4f}")

--- BẮT ĐẦU QUÁ TRÌNH TINH CHỈNH 20 CẤU HÌNH CHO DROPOUTNET ---
Trial [1/20] | Config: {'lr': 0.005, 'p': 0.3, 'lam': 0.1, 'wd': 0.0001} | NDCG@20 (Val): 0.0687
Trial [2/20] | Config: {'lr': 0.0001, 'p': 0.5, 'lam': 0.05, 'wd': 0.0001} | NDCG@20 (Val): 0.0609
Trial [3/20] | Config: {'lr': 0.0001, 'p': 0.1, 'lam': 0.1, 'wd': 1e-05} | NDCG@20 (Val): 0.0621
Trial [4/20] | Config: {'lr': 0.005, 'p': 0.8, 'lam': 0.5, 'wd': 1e-05} | NDCG@20 (Val): 0.0633
Trial [5/20] | Config: {'lr': 0.0005, 'p': 0.3, 'lam': 0.5, 'wd': 0.0001} | NDCG@20 (Val): 0.0540
Trial [6/20] | Config: {'lr': 0.0005, 'p': 0.3, 'lam': 0.01, 'wd': 0.001} | NDCG@20 (Val): 0.0539
Trial [7/20] | Config: {'lr': 0.0005, 'p': 0.1, 'lam': 0.5, 'wd': 1e-05} | NDCG@20 (Val): 0.0467
Trial [8/20] | Config: {'lr': 0.0001, 'p': 0.5, 'lam': 0.5, 'wd': 0.001} | NDCG@20 (Val): 0.0637
Trial [9/20] | Config: {'lr': 0.005, 'p': 0.8, 'lam': 0.1, 'wd': 0.001} | NDCG@20 (Val): 0.0583
Trial [10/20] | Config: {'lr': 0.0001, 'p': 0.5, 'lam': 0.01,

In [11]:
hist_cold_df = pd.concat([tr_cold, va_cold])
train_hist_dict = {}
for u, group in hist_cold_df.groupby('user_id'):
    u_str = str(u)
    if u_str in user2id:
        u_idx = user2id[u_str]
        item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
        if len(item_idxs) > 0:
            train_hist_dict[u_idx] = item_idxs
print(f'train_hist_dict sau khi gộp train_val với {len(train_hist_dict)} học viên.')

train_hist_dict sau khi gộp train_val với 98220 học viên.


In [12]:
seeds_list = [42, 2023, 2024, 1007, 999]

# Lưu kết quả cho tập (a) - Khóa lạnh và tập (b) - Toàn bộ item
results_a = []
results_b = []

print("\n--- BẮT ĐẦU ĐÁNH GIÁ 5 SEED CHO DROPOUTNET THỦ CÔNG ---")

for seed_val in seeds_list:
    set_seed(seed_val)
    
    model_seed = DropoutNetItemCold(d=64, d_content=Phi_V_tensor.shape[1], h=200).to(DEVICE)
    optimizer_seed = optim.Adam(model_seed.parameters(), lr=best_cfg['lr'], weight_decay=best_cfg['wd'])
    
    # Huấn luyện lại với seed cụ thể
    model_seed.train()
    for epoch in range(60):
        optimizer_seed.zero_grad()
        mask = (torch.rand(V_tensor.shape[0], 1, device=DEVICE) > best_cfg['p']).float()
        v_pref_masked = V_tensor * mask
        
        u_pred, v_pred = model_seed(U_tensor, v_pref_masked, Phi_V_tensor)
        orig_scores = torch.matmul(U_tensor, V_tensor.t())
        pred_scores = torch.matmul(u_pred, v_pred.t())
        
        loss = F.mse_loss(pred_scores, orig_scores)
        loss.backward()
        optimizer_seed.step()
        
    # Đánh giá sau khi huấn luyện xong
    model_seed.eval()
    with torch.no_grad():
        v_pref_cold_sim = torch.zeros_like(V_tensor)
        u_eval, v_eval = model_seed(U_tensor, v_pref_cold_sim, Phi_V_tensor)
        
        def score_fn_seed(users_tensor):
            u_idx = users_tensor.to(DEVICE)
            return torch.matmul(u_eval[u_idx], v_eval.t())
            
        # 1. Đánh giá trên tập (a) - Khóa lạnh
        test_res_a = evaluate(score_fn_seed, test_target_dict, mask=train_hist_dict, ks=(5, 10, 20), cand=cand_bool_mask_a, bs=1024)
        results_a.append(test_res_a)
        
        # 2. Đánh giá trên tập (b) - Toàn bộ item
        test_res_b = evaluate(score_fn_seed, test_target_dict, mask=train_hist_dict, ks=(5, 10, 20), cand=cand_bool_mask_b, bs=1024)
        results_b.append(test_res_b)


--- BẮT ĐẦU ĐÁNH GIÁ 5 SEED CHO DROPOUTNET THỦ CÔNG ---


In [13]:
print("\n========== KẾT QUẢ TỔNG HỢP DROPOUTNET (5 SEED) ==========")
print("--- Tập (a): Khóa lạnh ---")
for k in results_a[0].keys():
    seed_means = np.array([res[k].mean() for res in results_a])
    mean_value = seed_means.mean()
    std_value = seed_means.std(ddof=1)  
    print(f"{k}: {mean_value:.4f} ± {std_value:.4f}")

print("\n--- Tập (b): Toàn bộ item ---")
for k in results_b[0].keys():
    seed_means = np.array([res[k].mean() for res in results_b])
    mean_value = seed_means.mean()
    std_value = seed_means.std(ddof=1)  
    print(f"{k}: {mean_value:.4f} ± {std_value:.4f}")


========== KẾT QUẢ TỔNG HỢP DROPOUTNET (5 SEED) ==========
--- Tập (a): Khóa lạnh ---
Recall@5: 0.0909 ± 0.0088
Recall@10: 0.1497 ± 0.0097
Recall@20: 0.2201 ± 0.0067
NDCG@5: 0.0588 ± 0.0065
NDCG@10: 0.0792 ± 0.0051
NDCG@20: 0.0985 ± 0.0051
HR@5: 0.1201 ± 0.0082
HR@10: 0.1960 ± 0.0176
HR@20: 0.2859 ± 0.0123

--- Tập (b): Toàn bộ item ---
Recall@5: 0.0031 ± 0.0008
Recall@10: 0.0073 ± 0.0030
Recall@20: 0.0242 ± 0.0160
NDCG@5: 0.0017 ± 0.0006
NDCG@10: 0.0031 ± 0.0013
NDCG@20: 0.0074 ± 0.0046
HR@5: 0.0032 ± 0.0008
HR@10: 0.0079 ± 0.0032
HR@20: 0.0262 ± 0.0161
